In [ ]:
import pandas as pd
pd.__version__

In [ ]:
stock_kline = pd.read_csv("./stock_history_kline.csv")

In [ ]:
stock_kline.head()

In [ ]:
stock_kline.code.drop_duplicates().values

# 取出来601788的股票进行试验

In [ ]:
stock_kline_601788 = stock_kline[(stock_kline.code == 601788)]

stock_kline_601788.columns


# 根据当前的时间字段（time_key）提取出来日期（int）,并进行升序排列，以便进行MA计算

In [ ]:
stock_kline_601788["date_int"] = stock_kline_601788["time_key"].map(lambda x : int(str(x)[:10].replace('-','')))

print("获得当前的日期的int值\n",stock_kline_601788.date_int.head())

stock_kline_601788 = stock_kline_601788.sort_values(by='date_int' ,ascending=True)

print("获得当前的日期的int值\n",stock_kline_601788.date_int.head())

print("======================")

print(stock_kline_601788.head(10))

# 下一步计算通过MA函数进行MA计算

In [ ]:
print(stock_kline_601788.close.head(10))

In [ ]:
import numpy as np

def ma(n,data_frame):

    pos = 0
    ma_values = []
    for index,row in data_frame.iterrows():
        close_list = data_frame.close.values[pos-n+1:pos+1]
        pos = pos + 1 
    
        ma_N = np.average(close_list)
        
        ma_values.append(ma_N)
    
#         print("ma%d=" % (n), ma_N)
#         print(close_list)
#         print(index,row["close"],row["date_int"])
        
    data_frame["ma%d" % (n)] = ma_values
    return data_frame

stock_kline_601788 = ma(5, stock_kline_601788)
stock_kline_601788 = ma(10, stock_kline_601788)
stock_kline_601788 = ma(15, stock_kline_601788)
stock_kline_601788 = ma(20, stock_kline_601788)


        

In [ ]:
print(stock_kline_601788.head(5))
print(stock_kline_601788.tail(5))

In [ ]:
import datetime as dtime

stock_kline_601788["date"] = pd.to_datetime(stock_kline_601788['time_key'])
stock_kline_601788 = stock_kline_601788.set_index("date")

In [ ]:
stock_kline_601788.tail(10)

## 画出来一张K线、MA线图

In [ ]:
stock_kline_601788.tail(100).loc[:,["close","ma5","ma10",'ma15',"ma20"]].plot(x_compat=True, grid=True, figsize=(20,8))

### 策略说明
- 如果当前ma5>ma20 说明当前短期势头在上升
- 如果当前ma5<ma20 说明当前短期势头在下降

In [ ]:

cash_amount = 100000
stock_hold = 0

min_buy_unit = 100

stock_kline_601788['stock_hold'] = 0
stock_kline_601788['stock_bought'] = 0
stock_kline_601788['cash_amount'] = cash_amount
stock_kline_601788['total_capital'] = cash_amount

# 买的时候，跳开太高的数量
buy_cnt_jump_too_high = 0
# 买的时候，跳开太低的数量
buy_cnt_jump_too_low = 0

# 卖的时候，跳开太高的数量
sell_cnt_jump_too_high = 0
# 卖的时候，跳开太低的数量
sell_cnt_jump_too_low = 0

for index,row in stock_kline_601788.iterrows():
#         print("ma5=%.4f" % row['ma5'])
        ma5 = row['ma5']
        ma10 = row['ma10']
        ma15 = row['ma15']
        ma20 = row['ma20']
        last_close = row['last_close']
        open_price = row['open']
        close_price = row['close']
        time_day = index
        
        low_price = min(open_price, close_price)
        high_price = max(open_price, close_price)
        
        if ma5 > ma20:
            if ((ma5 - ma20) / ma20) > 0.01 :
#                 print("金叉")
#                 print("全部买入时机到time_day:%s" % time_day)
#                 print("ma5=%.4f , ma20 = %.4f" % (ma5,ma20))
                if cash_amount > 0 and cash_amount / last_close > min_buy_unit:
                    if last_close >= low_price and last_close <= high_price:
#                         print("last_close price is between open and close")
                        stock_can_buy = cash_amount / last_close
                        stock_bought = stock_can_buy -( stock_can_buy % min_buy_unit)
                        stock_hold = stock_hold + stock_bought
                        cash_amount = cash_amount - stock_bought * last_close
#                         print("stock_hold:%d, and cash amount:%.2f, and total_captical:%.2f" % (stock_hold, cash_amount, stock_hold*close_price+cash_amount))
                        stock_kline_601788.stock_bought[index] = stock_bought
                    elif last_close > high_price:
#                       昨日收盘价，高于等于今日最高价。则挂限价单不会成功。
                        print("jump too low, can't buy!!!!!!!!!")
                        buy_cnt_jump_too_low = buy_cnt_jump_too_low + 1
# #                       昨日收盘价高于今日的开盘价和收盘价，则按照开盘价较低买入。（策略在实现时，则要先查盘，再挂单）
                        price_to_buy = min(open_price, last_close)
                        print("jump too low , buy price is %.2f , open is %.2f, last_close is %.2f" % (price_to_buy, open_price, last_close))
                        stock_can_buy = cash_amount / price_to_buy
                        stock_bought = stock_can_buy -( stock_can_buy % min_buy_unit)
                        stock_hold = stock_hold + stock_bought
                        cash_amount = cash_amount - stock_bought * price_to_buy
                        print("stock_hold:%d, and cash amount:%.2f, and total_captical:%.2f" % (stock_hold, cash_amount, stock_hold*close_price+cash_amount))
                        stock_kline_601788.stock_bought[index] = stock_bought    
                    elif last_close < low_price:
#                       昨日收盘价，低于今日最低价，则挂单买入，也无法成功，价格太低。
                        print("jump too high, can't buy!!!!!!!!!")
                        buy_cnt_jump_too_high = buy_cnt_jump_too_high + 1
                    else:
                        print("WTF")
                        
        else:
            if ((ma20 - ma5) / ma5) > 0.01:
#                 print("死叉")
#                 print("全部卖出时机到")
#                 print("全部买入时机到time_day:%s" % time_day)
#                 print("ma5=%.4f , ma20 = %.4f" % (ma5,ma20))
                if stock_hold > 0 and stock_hold > min_buy_unit :
                    if last_close >= low_price and last_close <= high_price:
                        stock_can_sell = stock_hold
                        stock_sold= stock_can_sell -( stock_can_sell % min_buy_unit)
                        stock_hold = stock_hold - stock_sold
                        cash_amount = cash_amount + stock_sold * last_close
#                         print("stock_hold:%d, and cash amount:%.2f, and total_captical:%.2f" % (stock_hold, cash_amount, stock_hold*close_price+cash_amount))
                        stock_kline_601788.stock_bought[index] = -stock_hold
                    elif last_close < low_price:
#                       卖出日遇到了跳开，则可以通过开盘高价卖出,策略实现时，要能够先盯盘，看开盘价再挂单
                        price_to_sell = max(last_close , open_price)
                        print("time_key:%s" % time_day)
                        print("jump too high , sell price is %.2f , open is %.2f, last_close is %.2f" % (price_to_sell, open_price, last_close))
                        stock_can_sell = stock_hold
                        stock_sold= stock_can_sell -( stock_can_sell % min_buy_unit)
                        stock_hold = stock_hold - stock_sold
                        cash_amount = cash_amount + stock_sold * price_to_sell
                        print("stock_hold:%d, and cash amount:%.2f, and total_captical:%.2f" % (stock_hold, cash_amount, stock_hold*close_price+cash_amount))
                        stock_kline_601788.stock_bought[index] = -stock_hold
                        print("jump high, sell a high price")
                        sell_cnt_jump_too_high = sell_cnt_jump_too_high + 1
                        
                    elif last_close > high_price:
                        print("jump too low, can't sell!!!!!!!!!")
                        sell_cnt_jump_too_low = sell_cnt_jump_too_low + 1
                    else:
                        print("WTF")
                        

        stock_kline_601788.stock_hold[index] = stock_hold
        stock_kline_601788.cash_amount[index] = cash_amount
        stock_kline_601788.total_capital[index] = stock_hold*close_price+cash_amount

print("buy jump too high cnt: %d" % buy_cnt_jump_too_high)
print("buy jump too low cnt: %d" % buy_cnt_jump_too_low)
print("sell jump too high cnt: %d" % sell_cnt_jump_too_high)
print("sell jump too low cnt: %d" % sell_cnt_jump_too_low)

stock_kline_601788.tail()

In [ ]:
# help(stock_kline_601788.plot)

In [ ]:
stock_kline_601788.loc[:,["close","ma5","ma20",'total_capital','stock_hold','cash_amount']].plot(x_compat=True, grid=True, figsize=(20,8), secondary_y = ['total_capital','stock_hold','cash_amount'])
# stock_kline_601788.tail(1000).loc[:,['stock_hold','cash_amount','total_capital']].plot(secoundary_y=True)

In [ ]:
stock_kline_601788.loc[:,['stock_hold','cash_amount','total_capital']].plot(x_compat=True, grid=True, figsize=(20,8))

In [ ]:
stock_kline_601788.tail(1000).loc[:,["close","ma5","ma20",'total_capital','stock_hold','cash_amount']].plot(x_compat=True, grid=True, figsize=(20,8), secondary_y = ['total_capital','stock_hold','cash_amount'])
# stock_kline_601788.tail(1000).loc[:,['stock_hold','cash_amount','total_capital']].plot(secoundary_y=True)